# Serial 5 µL stock addition + 1500 µL DI water fill (Science Jubilee)

**Deck layout (edit slots to match your bed):**

| Slot | Labware |
|---|---|
| 0 | Tip rack for small pipette (5 µL steps) |
| 1 | Trash |
| 2 | Sample holder A (48 positions) |
| 3 | Stocks: 12 x 20 mL vials. `stocks[0]` = stock solution, `stocks[1:]` = DI water |
| 4 | Tip rack for P1000 (water steps) |
| 5 | Sample holder B (48 positions) |

Items marked **TODO** are placeholders (file names, calibration coordinates, tool configs) you must replace with your own values.

In [ ]:
# ----------- Science Jubilee -------------
from science_jubilee import Machine as Jub
from science_jubilee.tools import Pipette
import time
import math

In [ ]:
SIMULATED = False   # set True to dry-run without hardware
jubilee = Jub.Machine(address='192.168.1.2', simulated=SIMULATED)

### LOAD DECK + LABWARE

In [ ]:
deck = jubilee.load_deck('lab_automation_deck.json')

def load_and_calibrate(json_name, slot, upper_left, upper_right, lower_right):
    """Load labware into a slot and apply the 3-point manual offset."""
    lw = jubilee.load_labware(json_name, slot)
    lw.manual_offset([upper_left, upper_right, lower_right])
    lw.load_manualOffset()
    return lw

# ---- Trash (1) ----
trash = jubilee.load_labware('agilent_1_reservoir_290ml.json', 1)

# ---- Tip racks (2) ----
# TODO: replace calibration coordinates with your own measured points
tiprack_small = load_and_calibrate('opentrons_96_tiprack_300ul.json', 0,   # TODO: tip rack file for the small-volume pipette
                                   (26.2, 82.5), (125.6, 82.5), (125.8, 19.4))
tiprack_p1000 = load_and_calibrate('opentrons_96_tiprack_1000ul.json', 4,  # TODO: 1000 uL tip rack file
                                   (0.0, 0.0), (0.0, 0.0), (0.0, 0.0))

# ---- Stocks: 12 x 20 mL vials (index 0 = stock, 1..11 = DI water) ----
stocks = load_and_calibrate('beaker_stir_plate.json', 3,
                            (167.4, 176.8), (258.0, 176.8), (258.0, 119.8))

# ---- Sample holders (2 x 48) ----
# TODO: replace JSON name + calibration coordinates for your 48-position holders
holder_A = load_and_calibrate('septavialrev1_44_holder_2000ul.json', 2,   # TODO
                              (0.0, 0.0), (0.0, 0.0), (0.0, 0.0))
holder_B = load_and_calibrate('septavialrev1_44_holder_2000ul.json', 5,   # TODO
                              (0.0, 0.0), (0.0, 0.0), (0.0, 0.0))

deck.safe_z = 66.0

### EXPERIMENT SETTINGS

In [ ]:
# ---- Samples ----
WELLS_PER_HOLDER = 48
N_SAMPLES = 48        # how many samples to fill (max 96 with two holders)
# Samples are filled in order: all of holder A first, then holder B
sample_wells = [holder_A[i] for i in range(WELLS_PER_HOLDER)] + \
               [holder_B[i] for i in range(WELLS_PER_HOLDER)]
sample_wells = sample_wells[:N_SAMPLES]

# ---- Stock addition ----
STOCK_VOLUME_UL   = 5       # uL of stock[0] per sample
STOCK_INTERVAL_S  = 60      # one sample every 60 s

# ---- Water addition ----
WATER_VOLUME_UL   = 1500    # uL of DI water per sample
MAX_TRANSFER_UL   = 1000    # max volume the pipette can move at once
WATER_VIAL_START_UL = 18000 # TODO: how much water you actually loaded in each vial (vial max = 18000 uL)
WATER_DEAD_VOL_UL   = 1000  # leave this much in each vial to avoid aspirating air

# ---- Heights (mm above labware bottom) ----
ASPIRATE_HEIGHT  = 3
DISPENSE_HEIGHT  = 27       # TODO: tune for your 48-position holder

### LOAD TOOLS

In [ ]:
# TODO: tool indices and config files must match your tool changer setup
# Small-volume pipette (for 5 uL) -- 5 uL is below the usable range of a P300/P1000
'''
P_small = Pipette.Pipette.from_config(3, 'Pipette_small', 'P20_config.json')
jubilee.load_tool(P_small)
P_small.add_tiprack(tiprack_small)
P_small.trash = trash[0]


# P1000 (for water)
P1000 = Pipette.Pipette.from_config(4, 'Pipette_1000', 'P1000_config.json')
jubilee.load_tool(P1000)
P1000.add_tiprack(tiprack_p1000)
P1000.trash = trash[0]
'''

P300 = Pipette.Pipette.from_config(3,'Pipette','P300_config.json')
jubilee.load_tool(P300)

P300.add_tiprack(tiprack_small)
P300.trash = trash[0]

### FUNCTION 1: 5 µL of `stocks[0]` into a new sample every minute

Uses a fixed schedule (`next_time += 60`) so the time spent pipetting does not accumulate as drift. Each sample starts exactly 60 s after the previous one began.

In [ ]:
def transfer_stock_to_sample(volume, sample_index):
    """Move `volume` uL from stocks[0] to sample_wells[sample_index]."""
    P300.aspirate(volume, stocks[0].bottom(+ASPIRATE_HEIGHT))
    P300.dispense(volume, sample_wells[sample_index].bottom(+DISPENSE_HEIGHT))
    P300.blowout()


def timed_stock_addition(volume=STOCK_VOLUME_UL, interval_s=STOCK_INTERVAL_S, n_samples=N_SAMPLES):
    """Add `volume` uL of stocks[0] to each sample, one new sample every `interval_s` seconds."""
    jubilee.pickup_tool(P300)
    P300.pickup_tip(tiprack_small[0])   # one tip for the run (dispense is above the liquid, no contact)

    start = time.monotonic()
    for i in range(n_samples):
        target_time = start + i * interval_s
        wait = target_time - time.monotonic()
        if wait > 0:
            time.sleep(wait)
        t0 = time.monotonic()
        transfer_stock_to_sample(volume, i)
        print(f"Sample {i + 1}/{n_samples}: {volume} uL dispensed "
              f"(t = {t0 - start:.0f} s, took {time.monotonic() - t0:.1f} s)")

    P300.return_tip(tiprack_small[0])

### FUNCTION 2: 1500 µL DI water (`stocks[1:]`) into every sample

The P1000 can only move 1000 µL at a time, so `transfer_water` splits each sample's volume into equal chunks of ≤ 1000 µL (1500 µL → 2 x 750 µL). `next_water_vial` tracks how much liquid is left and moves on to the next vial in `stocks[1:]` when one runs low.

In [ ]:
# Remaining volume in each water vial (indices 1..11 of stocks)
water_vial_remaining = {i: WATER_VIAL_START_UL for i in range(1, len(stocks))}


def split_volume(total, max_per_transfer=MAX_TRANSFER_UL):
    """Split `total` uL into equal chunks, none larger than `max_per_transfer`.
    e.g. 1500 -> [750.0, 750.0]"""
    n = math.ceil(total / max_per_transfer)
    return [total / n] * n


def next_water_vial(volume):
    """Return the first water vial (index >= 1) with enough liquid left for `volume`."""
    for idx, remaining in water_vial_remaining.items():
        if remaining - volume >= WATER_DEAD_VOL_UL:
            return idx
    raise RuntimeError("Out of DI water: refill stocks[1:] and reset water_vial_remaining.")


def transfer_water_chunk(volume, sample_index):
    """One aspirate/dispense of <= 1000 uL from a water vial into a sample."""
    vial = next_water_vial(volume)
    P1000.aspirate(volume, stocks[vial].bottom(+ASPIRATE_HEIGHT))
    P1000.dispense(volume, sample_wells[sample_index].bottom(+DISPENSE_HEIGHT))
    P1000.blowout()
    water_vial_remaining[vial] -= volume


def transfer_water(total_volume, sample_index):
    """Transfer `total_volume` uL of water into one sample, in chunks <= 1000 uL."""
    for chunk in split_volume(total_volume):
        transfer_water_chunk(chunk, sample_index)


def water_addition(volume=WATER_VOLUME_UL, n_samples=N_SAMPLES):
    """Add `volume` uL of DI water to every sample."""
    jubilee.pickup_tool(P1000)
    P1000.pickup_tip()     # water only, so one tip for the whole run
    for i in range(n_samples):
        transfer_water(volume, i)
        print(f"Sample {i + 1}/{n_samples}: {volume} uL water added")
    P1000.drop_tip()

### RUN

In [ ]:
# Sanity check before running: water needed vs. water loaded
needed = N_SAMPLES * WATER_VOLUME_UL
available = sum(v - WATER_DEAD_VOL_UL for v in water_vial_remaining.values())
print(f"Water needed: {needed} uL | usable water loaded: {available} uL")
assert available >= needed, "Not enough water loaded in stocks[1:]"

In [ ]:
# Step 1: 5 uL of stock into a new sample every minute (~48 min for 48 samples)
timed_stock_addition()

In [ ]:
# Step 2: 1500 uL DI water into every sample
water_addition()

In [ ]:
jubilee.park_tool()